# Phase 8 — L'ordre des choses : découpage temporel

## Objectif général

Cette phase vise à évaluer le modèle sur des observations plus récentes que
celles utilisées pour l'entraînement.

Une découpe aléatoire mélange des signalements anciens et récents dans les jeux
d'entraînement et de test. Le modèle peut alors apprendre avec des informations
issues du futur par rapport à certains relevés du test.

La découpe temporelle évite ce problème : toutes les observations du jeu
d'entraînement sont plus anciennes que toutes les observations du jeu de test.

## 1. Imports des bibliothèques

In [17]:
from pathlib import Path
import csv
import re

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

## 2. Chemins, noms des colonnes et paramètres

Les résultats spécifiques à cette phase seront enregistrés dans le dossier :

```text
outputs/phase_8_decoupage_temporel/
```

In [18]:
DATA_PATH = Path("../data/releves_klaxo3.csv")

OUTPUT_DIR = Path("../outputs")
PHASE8_DIR = OUTPUT_DIR / "phase_8_decoupage_temporel"
PHASE8_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

MOTS_CLES_CANULAR = [
    "hoax",
    "fake",
    "prank",
    "joke",
    "not real",
    "made up",
    "fraud",
]

TEST_SIZE = 0.20
RANDOM_STATE = 42

## 3. Chargement robuste des lignes structurées

Le CSV ne contient pas d'en-têtes et certaines lignes peuvent avoir une
structure invalide.

Seules les lignes possédant exactement 11 champs sont intégrées au DataFrame
principal. Les lignes ayant un nombre différent de champs sont conservées à part
afin qu'aucune donnée ne disparaisse silencieusement.

In [19]:
lignes_valides = []
lignes_problemes = []

with open(
    DATA_PATH,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append(
                {
                    "numero_ligne": numero_ligne,
                    "nb_champs": len(row),
                    "contenu": row,
                }
            )

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Nombre de lignes chargées : {len(df)}")
print(f"Nombre de lignes isolées : {len(lignes_problemes)}")

Nombre de lignes chargées : 88679
Nombre de lignes isolées : 196


## 4. Conversion des types

Les durées et les coordonnées sont converties en nombres. Les dates
d'observation et de publication sont converties au format date.

Les valeurs impossibles à convertir deviennent manquantes, mais aucune ligne
n'est supprimée.

In [20]:
for col in ["duration_seconds", "latitude", "longitude"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

for col in ["datetime", "date_posted"]:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration_seconds             float64
duration_hours_min            object
comments                      object
date_posted           datetime64[ns]
latitude                     float64
longitude                    float64
dtype: object

## 5. Création de la cible artificielle `is_hoax`

La cible est construite comme dans les phases précédentes.

Un signalement est étiqueté comme canular lorsqu'un de ses commentaires contient
au moins un mot-clé associé à une fraude, une mise en scène ou une plaisanterie.

Cette cible est une pseudo-étiquette : elle ne constitue pas une vérité terrain.

In [21]:
pattern_canular = "|".join(
    re.escape(mot)
    for mot in MOTS_CLES_CANULAR
)

df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(
        pattern_canular,
        regex=True,
        na=False,
    )
    .astype(int)
)

print("Répartition des classes :")
display(df["is_hoax"].value_counts())

print("\nRépartition des classes en pourcentage :")
display(
    df["is_hoax"]
    .value_counts(normalize=True)
    .mul(100)
    .round(3)
)

Répartition des classes :


is_hoax
0    87810
1      869
Name: count, dtype: int64


Répartition des classes en pourcentage :


is_hoax
0    99.02
1     0.98
Name: proportion, dtype: float64

## 6. Création des variables utilisables par le modèle

Le modèle utilise des informations disponibles lors du signalement :

- ville ;
- État ou région ;
- pays ;
- forme observée ;
- durée ;
- coordonnées ;
- année, mois et heure de l'observation.

La colonne `comments` est exclue des variables d'entrée, car elle a été utilisée
pour construire `is_hoax`. L'inclure dans le modèle créerait une fuite de
données.

La colonne `date_posted` est également exclue, car elle correspond à une étape
postérieure de publication ou de traitement du dossier.

In [22]:
df["observation_year"] = df["datetime"].dt.year
df["observation_month"] = df["datetime"].dt.month
df["observation_hour"] = df["datetime"].dt.hour

df["text_features_without_leakage"] = (
    "city " + df["city"].fillna("").astype(str)
    + " state " + df["state"].fillna("").astype(str)
    + " country " + df["country"].fillna("").astype(str)
    + " shape " + df["shape"].fillna("").astype(str)
)

df[
    [
        "datetime",
        "observation_year",
        "observation_month",
        "observation_hour",
        "text_features_without_leakage",
    ]
].head()

,datetime,observation_year,observation_month,observation_hour,text_features_without_leakage
0,1949-10-10 20:30:00,1949.0,10.0,20.0,city san marcos state tx country us shape cyli...
1,1949-10-10 21:00:00,1949.0,10.0,21.0,city lackland afb state tx country shape light
2,1955-10-10 17:00:00,1955.0,10.0,17.0,city chester (uk/england) state country gb sh...
3,1956-10-10 21:00:00,1956.0,10.0,21.0,city edna state tx country us shape circle
4,1960-10-10 20:00:00,1960.0,10.0,20.0,city kaneohe state hi country us shape light


## 7. Choix de la date pour la découpe temporelle

Deux dates sont disponibles :

- `datetime` : date et heure à laquelle le témoin a observé le phénomène ;
- `date_posted` : date à laquelle le signalement a été publié ou traité.

La colonne retenue est `datetime`. Elle représente le moment réel de
l'observation et permet de simuler une situation où le Bureau entraîne le
modèle avec des observations anciennes, puis l'utilise sur de futurs
signalements.

Les relevés sans `datetime` ne peuvent pas être placés chronologiquement. Ils
sont comptés et mis à part pour l'évaluation temporelle, mais ils restent
conservés dans le DataFrame initial.

In [23]:
nombre_dates_manquantes = int(
    df["datetime"].isna().sum()
)

df_temporel = df.loc[
    df["datetime"].notna()
].copy()

print(f"Nombre total de relevés : {len(df)}")
print(f"Nombre de relevés sans datetime : {nombre_dates_manquantes}")
print(f"Nombre de relevés utilisables temporellement : {len(df_temporel)}")

print("\nDate d'observation minimale :")
print(df_temporel["datetime"].min())

print("\nDate d'observation maximale :")
print(df_temporel["datetime"].max())

Nombre total de relevés : 88679
Nombre de relevés sans datetime : 1220
Nombre de relevés utilisables temporellement : 87459

Date d'observation minimale :
1906-11-11 00:00:00

Date d'observation maximale :
2014-05-08 18:45:00


## 8. Tri des observations et choix de la date de coupure

Les relevés avec une date d'observation valide sont triés du plus ancien au plus
récent.

Les 80 % de relevés les plus anciens forment le jeu d'entraînement. Les 20 %
de relevés les plus récents forment le jeu de test.

La date de coupure est définie à partir de la position correspondant à 80 % des
relevés dans le DataFrame trié.

In [24]:
df_temporel = df_temporel.sort_values(
    "datetime"
).copy()

position_coupure = int(
    len(df_temporel) * (1 - TEST_SIZE)
)

date_coupure = df_temporel.iloc[
    position_coupure
]["datetime"]

print(f"Position de coupure : {position_coupure}")
print(f"Date de coupure : {date_coupure}")

Position de coupure : 69967
Date de coupure : 2012-01-17 18:00:00


## 9. Construction des jeux d'entraînement et de test

- Le jeu d'entraînement contient les observations strictement antérieures à la
  date de coupure.
- Le jeu de test contient les observations égales ou postérieures à cette date.

Cette règle garantit qu'aucune observation future n'est présente dans le jeu
d'entraînement.

In [25]:
df_train_temporel = df_temporel.loc[
    df_temporel["datetime"] < date_coupure
].copy()

df_test_temporel = df_temporel.loc[
    df_temporel["datetime"] >= date_coupure
].copy()

print(f"Nombre de relevés dans le train : {len(df_train_temporel)}")
print(f"Nombre de relevés dans le test : {len(df_test_temporel)}")

print("\nDernière date dans le train :")
print(df_train_temporel["datetime"].max())

print("\nPremière date dans le test :")
print(df_test_temporel["datetime"].min())

Nombre de relevés dans le train : 69967
Nombre de relevés dans le test : 17492

Dernière date dans le train :
2012-01-17 17:35:00

Première date dans le test :
2012-01-17 18:00:00


## 10. Vérification de l'ordre temporel

Cette cellule vérifie formellement que toutes les observations du jeu
d'entraînement sont plus anciennes que toutes les observations du jeu de test.

In [26]:
derniere_date_train = df_train_temporel["datetime"].max()
premiere_date_test = df_test_temporel["datetime"].min()

print(f"Dernière observation du train : {derniere_date_train}")
print(f"Première observation du test : {premiere_date_test}")

assert derniere_date_train < premiere_date_test

print("\nVérification réussie : le train est strictement antérieur au test.")

Dernière observation du train : 2012-01-17 17:35:00
Première observation du test : 2012-01-17 18:00:00

Vérification réussie : le train est strictement antérieur au test.


## 11. Proportion de canulars dans chaque période

La proportion de canulars est calculée séparément dans le train et dans le test.

Cette comparaison est importante : si les proportions sont différentes, cela
signifie que la distribution de la cible a évolué dans le temps. Le problème à
prédire peut alors être différent entre les données anciennes et les données
récentes.

In [27]:
nombre_canulars_train = int(
    df_train_temporel["is_hoax"].sum()
)

nombre_canulars_test = int(
    df_test_temporel["is_hoax"].sum()
)

proportion_canulars_train = (
    df_train_temporel["is_hoax"].mean()
)

proportion_canulars_test = (
    df_test_temporel["is_hoax"].mean()
)

resume_decoupage_temporel = pd.DataFrame(
    [
        {
            "jeu": "train",
            "nombre_releves": len(df_train_temporel),
            "nombre_canulars": nombre_canulars_train,
            "proportion_canulars": proportion_canulars_train,
            "date_min": df_train_temporel["datetime"].min(),
            "date_max": df_train_temporel["datetime"].max(),
        },
        {
            "jeu": "test",
            "nombre_releves": len(df_test_temporel),
            "nombre_canulars": nombre_canulars_test,
            "proportion_canulars": proportion_canulars_test,
            "date_min": df_test_temporel["datetime"].min(),
            "date_max": df_test_temporel["datetime"].max(),
        },
    ]
)

resume_decoupage_temporel

,jeu,nombre_releves,nombre_canulars,proportion_canulars,date_min,date_max
0,train,69967,686,0.009805,1906-11-11 00:00:00,2012-01-17 17:35:00
1,test,17492,138,0.007889,2012-01-17 18:00:00,2014-05-08 18:45:00


## 12. Définition de la matrice de variables `X` et de la cible `y`

La colonne `is_hoax` est la cible à prédire.

Les entrées du modèle correspondent aux variables textuelles sans commentaire
et aux variables numériques disponibles lors de l'observation.

In [28]:
features_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
]

features_modele = [
    "text_features_without_leakage",
] + features_numeriques

X_train_temporel = df_train_temporel[
    features_modele
].copy()

X_test_temporel = df_test_temporel[
    features_modele
].copy()

y_train_temporel = df_train_temporel[
    "is_hoax"
].copy()

y_test_temporel = df_test_temporel[
    "is_hoax"
].copy()

print("Dimensions de X_train :", X_train_temporel.shape)
print("Dimensions de X_test :", X_test_temporel.shape)
print("Dimensions de y_train :", y_train_temporel.shape)
print("Dimensions de y_test :", y_test_temporel.shape)

Dimensions de X_train : (69967, 7)
Dimensions de X_test : (17492, 7)
Dimensions de y_train : (69967,)
Dimensions de y_test : (17492,)


## 13. Construction du pipeline de machine learning

Le pipeline comprend :

1. Une vectorisation TF-IDF des informations textuelles de localisation et de
   forme.
2. Une imputation médiane pour les valeurs numériques manquantes.
3. Une régression logistique pour prédire la classe `is_hoax`.

Le pipeline est entraîné uniquement avec les données anciennes du train. Ainsi,
le vocabulaire TF-IDF et les médianes sont appris sans utiliser les données
récentes du test.

In [29]:
preprocessing_temporel = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=10_000,
                ngram_range=(1, 2),
            ),
            "text_features_without_leakage",
        ),
        (
            "numerique",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="median"),
                    ),
                ]
            ),
            features_numeriques,
        ),
    ]
)

modele_temporel = Pipeline(
    steps=[
        ("preprocessing", preprocessing_temporel),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

modele_temporel

,steps,"[('preprocessing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('texte', ...), ('numerique', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 14. Entraînement du modèle sur les observations anciennes

Le modèle apprend uniquement à partir du jeu d'entraînement temporel.

In [30]:
modele_temporel.fit(
    X_train_temporel,
    y_train_temporel,
)

print("Entraînement terminé.")

Entraînement terminé.


## 15. Prédiction sur les observations récentes

Le modèle est maintenant appliqué aux relevés du jeu de test, qui correspondent
aux observations les plus récentes du fichier.

In [31]:
y_pred_temporel = modele_temporel.predict(
    X_test_temporel
)

print("Répartition des prédictions :")
display(
    pd.Series(y_pred_temporel)
    .value_counts()
)

Répartition des prédictions :


0    12152
1     5340
Name: count, dtype: int64

## 16. Calcul des métriques

Les deux métriques principales demandées sont :

- **Recall** : parmi les canulars réellement présents dans le jeu de test,
  combien sont détectés.
- **Precision** : parmi les relevés signalés comme canulars par le modèle,
  combien sont réellement des canulars selon la règle retenue.

L'accuracy est calculée en complément, mais ne suffit pas à elle seule car les
canulars sont rares.

In [32]:
precision_temporelle = precision_score(
    y_test_temporel,
    y_pred_temporel,
    zero_division=0,
)

recall_temporel = recall_score(
    y_test_temporel,
    y_pred_temporel,
    zero_division=0,
)

accuracy_temporelle = accuracy_score(
    y_test_temporel,
    y_pred_temporel,
)

print(f"Precision temporelle : {precision_temporelle:.2%}")
print(f"Recall temporel : {recall_temporel:.2%}")
print(f"Accuracy temporelle : {accuracy_temporelle:.2%}")

Precision temporelle : 1.25%
Recall temporel : 48.55%
Accuracy temporelle : 69.45%


## 17. Matrice de confusion

La matrice de confusion permet de compter :

- les vrais négatifs : non-canular correctement reconnu ;
- les faux positifs : non-canular classé comme canular ;
- les faux négatifs : canular non détecté ;
- les vrais positifs : canular correctement détecté.

In [33]:
matrice_temporelle = confusion_matrix(
    y_test_temporel,
    y_pred_temporel,
)

df_matrice_temporelle = pd.DataFrame(
    matrice_temporelle,
    index=[
        "Réel : non-canular",
        "Réel : canular",
    ],
    columns=[
        "Prédit : non-canular",
        "Prédit : canular",
    ],
)

df_matrice_temporelle

,Prédit : non-canular,Prédit : canular
Réel : non-canular,12081,5273
Réel : canular,71,67


## 18. Rapport de classification détaillé

Le rapport affiche, pour chacune des deux classes, la precision, le recall et le
score F1.

In [34]:
print(
    classification_report(
        y_test_temporel,
        y_pred_temporel,
        target_names=[
            "non-canular",
            "canular",
        ],
        zero_division=0,
    )
)

              precision    recall  f1-score   support

 non-canular       0.99      0.70      0.82     17354
     canular       0.01      0.49      0.02       138

    accuracy                           0.69     17492
   macro avg       0.50      0.59      0.42     17492
weighted avg       0.99      0.69      0.81     17492



## 19. Comparaison avec la phase 7

La phase 7 a utilisé une découpe par événements. La phase 8 ajoute une
contrainte temporelle : le test contient exclusivement des observations plus
récentes que le train.

Les résultats de la phase 7 sont repris pour observer l'évolution des métriques
lorsque les conditions d'évaluation deviennent plus réalistes.

In [35]:
comparaison_phase7_phase8 = pd.DataFrame(
    [
        {
            "evaluation": "Phase 7 : découpe par événements",
            "precision": 0.012717852096090438,
            "recall": 0.49390243902439024,
        },
        {
            "evaluation": "Phase 8 : découpe temporelle",
            "precision": precision_temporelle,
            "recall": recall_temporel,
        },
    ]
)

comparaison_phase7_phase8

,evaluation,precision,recall
0,Phase 7 : découpe par événements,0.012718,0.493902
1,Phase 8 : découpe temporelle,0.012547,0.485507


## 20. Export des résultats

Les tableaux générés dans cette phase sont sauvegardés dans le dossier :

```text
outputs/phase_8_decoupage_temporel/
```

In [36]:
resume_decoupage_temporel.to_csv(
    PHASE8_DIR / "resume_decoupage_temporel.csv",
    index=False,
)

df_matrice_temporelle.to_csv(
    PHASE8_DIR / "matrice_confusion_temporelle.csv",
    index=True,
)

comparaison_phase7_phase8.to_csv(
    PHASE8_DIR / "comparaison_phase7_phase8.csv",
    index=False,
)

resultats_modele_temporel = pd.DataFrame(
    [
        {
            "modele": "Modèle sans fuite avec découpe temporelle",
            "date_coupure": date_coupure,
            "precision": precision_temporelle,
            "recall": recall_temporel,
            "accuracy": accuracy_temporelle,
            "nombre_train": len(df_train_temporel),
            "nombre_test": len(df_test_temporel),
            "proportion_canulars_train": proportion_canulars_train,
            "proportion_canulars_test": proportion_canulars_test,
            "releves_sans_datetime": nombre_dates_manquantes,
        }
    ]
)

resultats_modele_temporel.to_csv(
    PHASE8_DIR / "resultats_modele_temporel.csv",
    index=False,
)

print("Fichiers exportés dans :")
print(PHASE8_DIR)

Fichiers exportés dans :
..\outputs\phase_8_decoupage_temporel
